# Kimi K3 Direct — P11 evaluation extraction

This notebook runs two independent Kimi K3 annotations for every dialogue in the single saved evaluation set and scores each run against its gold annotations. Numbered cache records are isolated from training and validation cache under:

```text
extension/artifacts/extraction_cache/evaluation/moonshot-direct__kimi-k3-max/{prompt}/{dialogue_id}_{run}.json
```

The default prompt is P11. Valid numbered records are reused; missing or invalid records are requested again. Set `FORCE_RERUN=True` only when all numbered records should be regenerated and replaced.

## 1. Setup

In [10]:
import json
import os
import sys
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from IPython.display import display

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'extension' / 'artifacts').exists():
        os.chdir(_candidate)
        break
else:
    raise FileNotFoundError('Run this notebook from inside the repository.')
sys.path.insert(0, str(Path.cwd()))

from extension.scripts.data_management.load_annotation_data import load_dataset
from extension.scripts.annotation import extraction, moonshot_kimi, scoring

try:
    moonshot_kimi._api_key()
    KEY_FOUND = True
except RuntimeError:
    KEY_FOUND = False

print('repository:', Path.cwd())
print('MOONSHOT_API_KEY found:', KEY_FOUND)

repository: /Users/tandon.utsav2/Desktop/Experiment_1
MOONSHOT_API_KEY found: True


## 2. Load the evaluation set

There is one saved evaluation CSV. It contains both the model-visible dialogue content and the completed gold annotations used only by the scorer. Gold labels, rationale, adjudication, and thread metadata are never included in the model prompt.

In [11]:
EVALUATION_PATH = Path(
    'extension/artifacts/annotation_dev_val_and_eval_sets/evaluation_set.csv'
)

gold = load_dataset(EVALUATION_PATH)
DIALOGUES = extraction.dialogues_from(gold, split='evaluation')
DIALOGUE_IDS = [dialogue['dialogue_id'] for dialogue in DIALOGUES]

assert len(DIALOGUE_IDS) == len(set(DIALOGUE_IDS))
assert len(DIALOGUE_IDS) == 36
print(f'{len(DIALOGUES)} evaluation dialogues, {len(gold)} scored units')
print('dialogue IDs:', DIALOGUE_IDS)

36 evaluation dialogues, 250 scored units
dialogue IDs: [41, 48, 105, 141, 158, 263, 296, 413, 446, 496, 508, 518, 570, 588, 623, 642, 650, 693, 844, 985, 1019, 1220, 1348, 1407, 1492, 1497, 1502, 1574, 1652, 1659, 1708, 1793, 1933, 2070, 2182, 2234]


## 3. Experiment and cache configuration

The cache redirection below exists only in this notebook process; it does not modify the shared extraction, model, or scoring scripts. Each worker receives a thread-local run number so concurrent calls write to distinct `_1.json` and `_2.json` files.

In [12]:
PROMPT = 'P11'
RUN_NUMBERS = (1, 2)
EFFORT = 'max'
MAX_WORKERS = 20
FORCE_RERUN = False
SLUG = moonshot_kimi.cache_slug(EFFORT)
EVALUATION_CACHE_ROOT = Path('extension/artifacts/extraction_cache/evaluation')
_cache_context = threading.local()


def evaluation_run_path(model_slug, prompt_name, dialogue_id, run_number):
    model_folder = model_slug.replace('/', '__').replace(':', '-')
    folder = EVALUATION_CACHE_ROOT / model_folder / prompt_name
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f'{dialogue_id}_{run_number}.json'


def evaluation_cache_path(model_slug, prompt_name, dialogue_id, split='evaluation'):
    run_number = getattr(_cache_context, 'run_number', None)
    if run_number is None:
        raise RuntimeError('Evaluation cache path requested outside a numbered run.')
    return evaluation_run_path(model_slug, prompt_name, dialogue_id, run_number)


def evaluation_cached_ok(model_slug, prompt_name, dialogue_id, split='evaluation'):
    path = evaluation_cache_path(model_slug, prompt_name, dialogue_id, split)
    if FORCE_RERUN or not path.exists():
        return False
    try:
        return bool(json.loads(path.read_text()).get('valid', False))
    except (OSError, json.JSONDecodeError):
        return False


# Runtime-only redirection for extraction and scoring.
moonshot_kimi.cache_path = evaluation_cache_path
moonshot_kimi.cached_ok = evaluation_cached_ok
scoring.cache_path = evaluation_cache_path

print('model:', SLUG)
print('prompt:', PROMPT)
print('numbered runs:', RUN_NUMBERS)
print('max workers:', MAX_WORKERS)
print('force rerun:', FORCE_RERUN)
print('evaluation cache root:', EVALUATION_CACHE_ROOT)

model: moonshot-direct/kimi-k3-max
prompt: P11
numbered runs: (1, 2)
max workers: 20
force rerun: False
evaluation cache root: extension/artifacts/extraction_cache/evaluation


## 4. Run or reuse both numbered extractions

Every `(dialogue, run)` pair is an independent job. With the default `FORCE_RERUN=False`, valid copied cache records return `cached`; only missing or invalid records make API requests.

In [13]:
if FORCE_RERUN and not KEY_FOUND:
    raise RuntimeError('MOONSHOT_API_KEY is required when forcing extraction.')


def run_numbered_annotation(dialogue, run_number):
    _cache_context.run_number = run_number
    try:
        return moonshot_kimi.generate_annotation(PROMPT, dialogue, EFFORT)
    finally:
        del _cache_context.run_number


jobs = [
    (dialogue, run_number)
    for dialogue in DIALOGUES
    for run_number in RUN_NUMBERS
]
run_rows = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {
        pool.submit(run_numbered_annotation, dialogue, run_number): (
            dialogue['dialogue_id'], run_number
        )
        for dialogue, run_number in jobs
    }
    for future in as_completed(futures):
        dialogue_id, run_number = futures[future]
        try:
            status = future.result()
        except Exception as exc:
            status = f'error: {exc}'
        run_rows.append({
            'dialogue_id': dialogue_id, 'run': run_number, 'status': status
        })
        print(f'{PROMPT}/{dialogue_id}_{run_number}: {status}')

run_results = pd.DataFrame(run_rows).sort_values(['run', 'dialogue_id'])
display(pd.crosstab(run_results['run'], run_results['status']))

P11/508_1: cached
P11/141_1: cached
P11/48_2: cached
P11/41_1: cached
P11/296_2: cached
P11/263_2: cached
P11/588_1: cached
P11/296_1: cached
P11/446_2: cached
P11/158_1: cached
P11/105_1: cached
P11/496_1: cached
P11/263_1: cached
P11/518_2: cached
P11/518_1: cached
P11/105_2: cached
P11/623_1: cached
P11/413_1: cached
P11/446_1: cached
P11/413_2: cached
P11/48_1: cached
P11/158_2: cached
P11/588_2: cached
P11/41_2: cached
P11/496_2: cached
P11/141_2: cached
P11/508_2: cached
P11/570_1: cached
P11/570_2: cached
P11/693_1: cached
P11/642_2: cached
P11/693_2: cached
P11/650_2: cached
P11/650_1: cached
P11/844_1: cached
P11/623_2: cached
P11/844_2: cached
P11/1407_1: cached
P11/985_1: cached
P11/1348_2: cached
P11/642_1: cached
P11/1019_1: cached
P11/1492_2: cached
P11/1497_1: cached
P11/1220_1: cached
P11/1348_1: cached
P11/985_2: cached
P11/1220_2: cached
P11/1019_2: cached
P11/1407_2: cached
P11/1492_1: cached
P11/1497_2: cached
P11/1502_1: cached
P11/2070_2: cached
P11/1708_1: cached

status,cached
run,
1,36
2,36


## 5. Inspect numbered-cache coverage

Coverage is reported separately for each run. Invalid and missing IDs are printed explicitly so they can be rerun without disturbing valid records.

In [14]:
coverage_rows = []
for run_number in RUN_NUMBERS:
    valid_ids, invalid_ids, missing_ids, unreadable_ids = [], [], [], []
    for dialogue_id in DIALOGUE_IDS:
        path = evaluation_run_path(SLUG, PROMPT, dialogue_id, run_number)
        if not path.exists():
            missing_ids.append(dialogue_id)
            continue
        try:
            record = json.loads(path.read_text())
        except (OSError, json.JSONDecodeError):
            unreadable_ids.append(dialogue_id)
            continue
        (valid_ids if record.get('valid') else invalid_ids).append(dialogue_id)

    coverage_rows.append({
        'run': run_number,
        'expected': len(DIALOGUE_IDS),
        'valid': len(valid_ids),
        'invalid': len(invalid_ids),
        'missing': len(missing_ids),
        'unreadable': len(unreadable_ids),
        'valid_rate': len(valid_ids) / len(DIALOGUE_IDS),
        'invalid_ids': invalid_ids,
        'missing_ids': missing_ids,
        'unreadable_ids': unreadable_ids,
    })

coverage = pd.DataFrame(coverage_rows).set_index('run')
display(coverage[['expected', 'valid', 'invalid', 'missing', 'unreadable', 'valid_rate']])
issues = coverage[['invalid_ids', 'missing_ids', 'unreadable_ids']].map(bool).any(axis=1)
if issues.any():
    display(coverage.loc[issues, ['invalid_ids', 'missing_ids', 'unreadable_ids']])

,expected,valid,invalid,missing,unreadable,valid_rate
run,,,,,,
1,36,36,0,0,0,1.0
2,36,36,0,0,0,1.0


## 6. Score each evaluation run

Each numbered run is scored independently against all 36 gold evaluation dialogues. Agreement and classification metrics use valid records only; `valid_rate` retains all 36 expected dialogues in its denominator.

In [15]:
score_rows = []
for run_number in RUN_NUMBERS:
    _cache_context.run_number = run_number
    try:
        metrics = scoring.score_config(
            gold=gold,
            model_slug=SLUG,
            prompt_name=PROMPT,
            dids=DIALOGUE_IDS,
            n_boot=0,
            split='evaluation',
        )
    finally:
        del _cache_context.run_number
    metrics['run'] = run_number
    score_rows.append(metrics)

scores = pd.DataFrame(score_rows).set_index('run').sort_index()
print(f'Scored {len(scores)} evaluation runs.')

Scored 2 evaluation runs.


### 6.1 Overall five-family results

In [16]:
overall_columns = [
    'prompt_word_count', 'valid_rate',
    'macro_f1_P', 'micro_f1_P', 'weighted_f1_P',
    'accuracy', 'alpha', 'kappa', 'latency_s',
]
overall_results = scores[overall_columns].rename(columns={
    'prompt_word_count': 'prompt words',
    'valid_rate': 'valid rate',
    'macro_f1_P': 'macro F1 (P)',
    'micro_f1_P': 'micro F1 (P)',
    'weighted_f1_P': 'weighted F1 (P)',
    'latency_s': 'mean latency (s)',
})
display(overall_results.round(3))

,prompt words,valid rate,macro F1 (P),micro F1 (P),weighted F1 (P),accuracy,alpha,kappa,mean latency (s)
run,,,,,,,,,
1,45617,1.0,0.870,0.893,0.906,0.959,0.881,0.881,837.170
2,45617,1.0,0.937,0.924,0.925,0.960,0.881,0.881,851.617


### 6.2 Results by misconception family

In [17]:
family_f1 = scores[[f'f1_{family}' for family in scoring.FAMILIES]].copy()
family_f1.columns = list(scoring.FAMILIES)
family_accuracy = scores[[
    f'accuracy_{family}' for family in scoring.FAMILIES
]].copy()
family_accuracy.columns = list(scoring.FAMILIES)

print('F1 for detecting P')
display(family_f1.round(3))
print('Exact P/A/N accuracy')
display(family_accuracy.round(3))

F1 for detecting P


,comprehension,relevance,principles,wrong_operation,steps
run,,,,,
1,0.906,1.000,1.0,0.885,0.560
2,0.924,0.958,1.0,0.868,0.933


Exact P/A/N accuracy


,comprehension,relevance,principles,wrong_operation,steps
run,,,,,
1,0.928,0.996,0.984,0.960,0.928
2,0.928,0.988,0.996,0.928,0.960


### 6.3 Pooled conceptual/procedural results

Comprehension, relevance, and principles are pooled into conceptual; wrong operation and steps are pooled into procedural.

In [18]:
pooled_columns = [
    'pooled_macro_f1_P', 'pooled_micro_f1_P',
    'pooled_weighted_f1_P', 'pooled_accuracy', 'pooled_alpha',
    'pooled_f1_conceptual', 'pooled_f1_procedural',
    'pooled_accuracy_conceptual', 'pooled_accuracy_procedural',
]
pooled_results = scores[pooled_columns].rename(columns={
    'pooled_macro_f1_P': 'macro F1 (P)',
    'pooled_micro_f1_P': 'micro F1 (P)',
    'pooled_weighted_f1_P': 'weighted F1 (P)',
    'pooled_accuracy': 'accuracy',
    'pooled_alpha': 'alpha',
    'pooled_f1_conceptual': 'conceptual F1 (P)',
    'pooled_f1_procedural': 'procedural F1 (P)',
    'pooled_accuracy_conceptual': 'conceptual accuracy',
    'pooled_accuracy_procedural': 'procedural accuracy',
})
display(pooled_results.round(3))

,macro F1 (P),micro F1 (P),weighted F1 (P),accuracy,alpha,conceptual F1 (P),procedural F1 (P),conceptual accuracy,procedural accuracy
run,,,,,,,,,
1,0.853,0.887,0.895,0.916,0.850,0.926,0.779,0.916,0.916
2,0.907,0.920,0.921,0.924,0.863,0.932,0.882,0.920,0.928


## Notes

- The evaluation CSV is never passed wholesale to the model; prompting uses the dialogue and ordered unit allowlist created by `extraction.dialogues_from`.
- Run 1 and run 2 are independent model responses, not folds or subsets.
- `MAX_WORKERS` bounds concurrent missing/invalid requests; cached records return immediately.
- The extra copied `1497_3.json` is retained in the evaluation cache but deliberately excluded because this notebook scores only runs 1 and 2.
- Moonshot reports token usage but not dollar cost, so cost is not displayed.